In [2]:
import pandas as pd
import numpy as np

In [3]:
original_df = pd.read_csv(r"C:\Retail_pulse\RetailPulse-AI-Powered-Customer-Analytics-Demand-Forecasting\data\processed\cleaned_retail_data.csv", encoding="ISO-8859-1")
original_df.head()
original_df.shape

C:\Users\tusha\AppData\Local\Temp\ipykernel_17524\2010421478.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  original_df = pd.read_csv(r"C:\Retail_pulse\RetailPulse-AI-Powered-Customer-Analytics-Demand-Forecasting\data\processed\cleaned_retail_data.csv", encoding="ISO-8859-1")


(530104, 9)

In [26]:
original_df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'Revenue'],
      dtype='object')

# FEATURES WE WILL BE CREATING FOR CUSTOMER SEGMENTATION => 
1.) Recency → days since last purchase
2.) Frequency → unique invoice count
3.) avg_order_value → Monetary / Frequency
4.) avg_basket_size → avg items per invoice

# AS MOSTLY TRANSACTION ARE INSIDE UNITED KINGDOM . SO COUNTRY DOESN'T AFFECT SO MUCH . SO WE ENCODE COUNTRY COLUMN AS (UK OR NOT UK.)

In [20]:
modified_df = original_df.copy()

modified_df['InvoiceDate'] = pd.to_datetime(modified_df['InvoiceDate'])
time_period = modified_df['InvoiceDate'].max() + pd.Timedelta(days=1)
valid = modified_df[(modified_df['CustomerID'] != -1)]
rfm = valid.groupby('CustomerID').agg(
    Recency = ('InvoiceDate', lambda x: (time_period - x.max()).days),
    Frequency = ('InvoiceNo', 'nunique'),
    Monetary = ('Revenue','sum'),
    unique_products= ('StockCode', 'nunique'),
    lifespan_days= ('InvoiceDate', lambda x: (x.max() - x.min()).days),
    avg_purchase_interval= ('InvoiceDate', lambda x: (x.max() - x.min()).days / (x.nunique() - 1) if x.nunique() > 1 else 0),
    country= ('Country', lambda x: x.mode()[0]),
).reset_index()
inv = valid.groupby(['CustomerID','InvoiceNo']).agg(
    order_value = ('Revenue','sum'),
    basket_size = ('Quantity','sum'),
).reset_index()

inv_feats = inv.groupby('CustomerID').agg(
    avg_order_value = ('order_value','mean'),
    avg_basket_size = ('basket_size','mean'),
).reset_index()
rfm = rfm.merge(inv_feats, on='CustomerID', how='left')
print(f"NUMBER OF TRANSACTIONS INSIDE UNITED KINGDOM => {len(rfm[rfm['country'] == 'United Kingdom']):,}")
print(f"% OF TOTAL TRANSACTIONS INSIDE UNITED KINGDOM => {len(rfm[rfm['country'] == 'United Kingdom']) / len(rfm) * 100:.2f}%")

rfm['country']=rfm['country'].apply(lambda x: 1 if x == 'United Kingdom' else 0)
rfm['country']
churn_df=rfm.copy()

NUMBER OF TRANSACTIONS INSIDE UNITED KINGDOM => 3,920
% OF TOTAL TRANSACTIONS INSIDE UNITED KINGDOM => 90.36%


# FEATURES WE WILL BE CREATING FOR CHURN PREDICTION =>
1.) is_churned→ Recency > 90 (this is THE TARGET variable) (90 IS ARBITRARY . WE CAN DECIDE ON BASIS OF avg_purchase_interval column.)
2.) purchase_rate→ Frequency / lifespan_days  (how dense their buying is)
3.) is_single_purchase → Frequency == 1
4.) missed_expected_visits → Recency / avg_purchase_interval

In [21]:

threshold = churn_df['avg_purchase_interval'].mean() * 2
churn_df['is_churned'] = (churn_df['Recency'] > threshold).astype(int)
churn_df['purchase_rate'] = (
    churn_df['Frequency'] / (churn_df['lifespan_days'] + 1)
)
churn_df['is_single_purchase'] = (churn_df['Frequency'] == 1).astype(int)
churn_df['missed_expected_visits'] = (
    churn_df['Recency'] / (churn_df['avg_purchase_interval'] + 1)
)
churn_df

,CustomerID,Recency,Frequency,Monetary,unique_products,lifespan_days,avg_purchase_interval,country,avg_order_value,avg_basket_size,is_churned,purchase_rate,is_single_purchase,missed_expected_visits
0,12346,326,1,77183.60,1,0,0.000000,1,77183.600000,74215.000000,1,1.000000,1,326.000000
1,12347,3,7,4310.00,103,365,60.833333,0,615.714286,351.142857,0,0.019126,0,0.048518
2,12348,76,4,1797.24,22,283,94.333333,0,449.310000,585.250000,0,0.014085,0,0.797203
3,12349,19,1,1757.55,73,0,0.000000,0,1757.550000,631.000000,0,1.000000,1,19.000000
4,12350,311,1,334.40,17,0,0.000000,0,334.400000,197.000000,1,1.000000,1,311.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4333,18280,278,1,180.60,10,0,0.000000,1,180.600000,45.000000,1,1.000000,1,278.000000
4334,18281,181,1,80.82,7,0,0.000000,1,80.820000,54.000000,1,1.000000,1,181.000000
4335,18282,8,2,178.05,12,119,119.000000,1,89.025000,51.500000,0,0.016667,0,0.066667
4336,18283,4,16,2094.88,263,334,25.692308,1,130.930000,87.312500,0,0.047761,0,0.149856


#  REMOVING HIGHLY CORRELATED COLUMNS . 

In [22]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import pandas as pd
vif_cols = [
    'Frequency', 'Monetary', 'unique_products', 'lifespan_days',
    'avg_purchase_interval', 'country', 'avg_order_value',
    'avg_basket_size', 'purchase_rate', 'is_single_purchase',
    'missed_expected_visits'
]

In [24]:
def calculate_vif(df, cols):
    X = df[cols].dropna()
    return pd.DataFrame({
        'Feature': cols,
        'VIF'    : [variance_inflation_factor(X.values, i)
                    for i in range(len(cols))]
    }).sort_values('VIF', ascending=False)

cols = vif_cols.copy()

while True:
    vif = calculate_vif(churn_df, cols)
    max_vif = vif.iloc[0]
    print(vif.to_string(index=False))
    if max_vif['VIF'] > 8:
        print(f"\n❌ Dropping: {max_vif['Feature']} (VIF={max_vif['VIF']:.2f})\n")
        cols.remove(max_vif['Feature'])
    else:
        print(f"\n✅ All VIF < 8 — Final features: {cols}")
        break
cols.extend(["CustomerID","is_churned"])
churn_df=churn_df[[col for col in cols]]

               Feature      VIF
       avg_order_value 9.032512
       avg_basket_size 8.096110
         lifespan_days 5.469718
    is_single_purchase 5.305564
               country 4.980815
         purchase_rate 4.004688
             Frequency 3.857089
       unique_products 3.019086
 avg_purchase_interval 2.899581
missed_expected_visits 2.783378
              Monetary 2.015160

❌ Dropping: avg_order_value (VIF=9.03)

               Feature      VIF
         lifespan_days 5.462404
    is_single_purchase 5.302793
               country 4.978875
         purchase_rate 3.987398
             Frequency 3.753310
       unique_products 3.015621
 avg_purchase_interval 2.896974
missed_expected_visits 2.783378
              Monetary 1.767830
       avg_basket_size 1.196255

✅ All VIF < 8 — Final features: ['Frequency', 'Monetary', 'unique_products', 'lifespan_days', 'avg_purchase_interval', 'country', 'avg_basket_size', 'purchase_rate', 'is_single_purchase', 'missed_expected_visits']


In [25]:
churn_df

,Frequency,Monetary,unique_products,lifespan_days,avg_purchase_interval,country,avg_basket_size,purchase_rate,is_single_purchase,missed_expected_visits,CustomerID,is_churned
0,1,77183.60,1,0,0.000000,1,74215.000000,1.000000,1,326.000000,12346,1
1,7,4310.00,103,365,60.833333,0,351.142857,0.019126,0,0.048518,12347,0
2,4,1797.24,22,283,94.333333,0,585.250000,0.014085,0,0.797203,12348,0
3,1,1757.55,73,0,0.000000,0,631.000000,1.000000,1,19.000000,12349,0
4,1,334.40,17,0,0.000000,0,197.000000,1.000000,1,311.000000,12350,1
...,...,...,...,...,...,...,...,...,...,...,...,...
4333,1,180.60,10,0,0.000000,1,45.000000,1.000000,1,278.000000,18280,1
4334,1,80.82,7,0,0.000000,1,54.000000,1.000000,1,181.000000,18281,1
4335,2,178.05,12,119,119.000000,1,51.500000,0.016667,0,0.066667,18282,0
4336,16,2094.88,263,334,25.692308,1,87.312500,0.047761,0,0.149856,18283,0


# FEATUTRES FOR DEMAND FORECASTING => 
1.) SEASONALITY CHECKING (MONTHLY,WEEKLY)
2.) ROLLING MEAN (7 DAYS, 30 DAYS)
3.) LAGS (7 DAYS,30 DAYS)